# OneLake → Azure AI Search with Content Understanding + GenAI image verbalization (GA)

Builds an end-to-end pipeline, **fully on GA APIs**, that:

1. Reads files from a **Microsoft Fabric Lakehouse** (OneLake `Files/`).
2. Runs the **Content Understanding skill** to extract structured content with page numbers, **plus** binary figure data (`extractionOptions: ["images", "locationMetadata"]`).
3. Chains the **GenAI Prompt skill** (GA in `2026-04-01`) over each extracted figure to generate a natural-language description (image verbalization).
4. Embeds both **text chunks** and **figure descriptions** with **Azure OpenAI** and writes them to the **same** AI Search index.
5. Each search doc has a `kind` field (`text` or `figure`) so retrieval can fuse them naturally.

## Why this exists

The CU skill's built-in image verbalization (`modelName` / `modelDeployment`) is **preview only** (`2026-05-01-preview`). This notebook achieves the same result on GA today by chaining the GA GenAI Prompt skill onto CU's `normalized_images` output. See `README.md` in this folder for the full comparison.

## Prerequisites

- Azure AI Search service (Basic or higher) **in the same tenant** as your Fabric workspace.
- Search service has a **system-assigned managed identity** with **Contributor** on the Fabric workspace.
- Fabric admin: *Allow apps running outside of Fabric to access data via OneLake* is **enabled**.
- An **Azure AI Foundry resource** (provisioned in [ai.azure.com](https://ai.azure.com); ARM type `Microsoft.CognitiveServices/accounts`, kind `AIServices`) — this is what the Content Understanding skill bills against. It must be in a region that supports Content Understanding for the modality you use:
  - Americas: `westus`, `westus3`, `eastus`, `eastus2`, `southcentralus`
  - Europe: `northeurope`, `westeurope`, `swedencentral`, `uksouth`
  - Asia Pacific: `australiaeast`, `japaneast`, `southeastasia`
  - Authoritative matrix (per modality + API version): https://learn.microsoft.com/azure/ai-services/content-understanding/language-region-support
  - Audio / video / some prebuilt analyzers are in **fewer** regions than document — check the matrix if you plan to extend beyond documents.
  - Bind to it via **key** (quick) or **managed identity** (recommended; assign your search service MI the **Cognitive Services User** role on the Foundry resource).
- An **Azure OpenAI** resource with a `text-embedding-3-large` deployment.
- Files already uploaded to `Files/engagements/project001/raw_files` in the lakehouse (PDF / DOCX / PPTX work best for page numbers).

## Environment setup (run once, in a terminal)

Create a local virtual environment and register it as a Jupyter kernel so this notebook can use it.

**Windows / PowerShell**

```powershell
cd notebooks
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name fti-onelake --display-name "Python (fti-onelake)"
Copy-Item .env.example .env   # then edit .env with your values
```

**macOS / Linux**

```bash
cd notebooks
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user --name fti-onelake --display-name "Python (fti-onelake)"
cp .env.example .env          # then edit .env with your values
```

Then in this notebook, select the **Python (fti-onelake)** kernel (top-right kernel picker in VS Code, or *Kernel → Change kernel* in Jupyter Lab).

The `%pip install` cell below is a no-op once the venv is active — leave it in for Colab / Codespaces users, or skip it.

## 1. Install + import

In [ ]:
# Skip if you already installed via `pip install -r requirements.txt` in your venv.
%pip install --quiet -r requirements.txt

In [ ]:
import os, json, time, requests
from dotenv import load_dotenv
load_dotenv(override=True)

SEARCH_ENDPOINT  = os.environ['SEARCH_ENDPOINT']
SEARCH_ADMIN_KEY = os.environ['SEARCH_ADMIN_KEY']
API_VERSION      = '2026-04-01'                            # GA: CU + GenAI Prompt skill

FABRIC_WORKSPACE_GUID = os.environ['FABRIC_WORKSPACE_GUID']
LAKEHOUSE_GUID        = os.environ['LAKEHOUSE_GUID']
LAKEHOUSE_SUBPATH     = 'engagements/project001/raw_files'

# ---- Azure AI Foundry resource (billing target for Content Understanding) ----
# This is the Foundry resource you create in https://ai.azure.com
# (ARM type: Microsoft.CognitiveServices/accounts, kind: AIServices).
# Endpoint is on the resource's 'Keys and Endpoint' blade.
FOUNDRY_ENDPOINT = os.environ['FOUNDRY_ENDPOINT']           # https://<name>.cognitiveservices.azure.com
FOUNDRY_KEY      = os.environ.get('FOUNDRY_KEY')            # optional if using managed identity
FOUNDRY_RESOURCE_ID = os.environ.get('FOUNDRY_RESOURCE_ID') # /subscriptions/.../accounts/<name> — for identity binding

# ---- Chat model (GenAI Prompt skill) for image verbalization ----
# Re-uses AOAI_ENDPOINT + AOAI_KEY — chat and embedding deployments live in the same resource.
# Only the deployment name differs.
VERBALIZE_PROMPT = (
    'You are describing a figure extracted from a document for a search index. '
    'Describe what is shown concretely: chart type, axes, key values, labeled entities, '
    'relationships. Avoid speculation. 2-4 sentences.'
)

# Multimodal chat deployment (must support image input: gpt-4o, gpt-4.1, gpt-4o-mini, ...).
# Lives in the same AOAI / Foundry resource as the embedding deployment — re-uses AOAI_ENDPOINT + AOAI_KEY.
AOAI_CHAT_DEPL = os.environ.get('AOAI_CHAT_DEPL', 'gpt-4o-mini')

# ---- Azure OpenAI for embeddings ----
# AOAI_ENDPOINT can be either:
#   - the raw AOAI resource: https://<aoai-name>.openai.azure.com
#   - an APIM gateway in front of AOAI: https://<apim>.azure-api.net/<api-suffix>
# If using APIM, AOAI_KEY must be a value APIM accepts (subscription key or pass-through),
# and the APIM policy must forward to /openai/deployments/{deployment}/embeddings.
AOAI_ENDPOINT   = os.environ['AOAI_ENDPOINT']
AOAI_KEY        = os.environ['AOAI_KEY']
AOAI_EMBED_DEPL = os.environ.get('AOAI_EMBED_DEPL', 'text-embedding-3-small')
AOAI_EMBED_MODEL= os.environ.get('AOAI_EMBED_MODEL', 'text-embedding-3-small')
EMBED_DIMS      = int(os.environ.get('EMBED_DIMS', '1536'))  # small=1536, large=3072

PREFIX    = 'fti-onelake-cu-verb'
DS_NAME   = f'{PREFIX}-ds'
SS_NAME   = f'{PREFIX}-ss'
IDX_NAME  = f'{PREFIX}-idx'
IDXR_NAME = f'{PREFIX}-idxr'

HEADERS = {'Content-Type': 'application/json', 'api-key': SEARCH_ADMIN_KEY}

def put(path, body):
    r = requests.put(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=HEADERS, data=json.dumps(body))
    if r.status_code >= 300: raise RuntimeError(f'{r.status_code}: {r.text}')
    return r
def post(path):
    r = requests.post(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=HEADERS)
    if r.status_code >= 300: raise RuntimeError(f'{r.status_code}: {r.text}')
    return r
def get(path):
    return requests.get(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=HEADERS).json()

print('Config loaded.')

ModuleNotFoundError: No module named 'requests'

## 2. OneLake data source

**Note:** the OneLake indexer does *not* accept the `https://onelake.dfs.fabric.microsoft.com/...` URL. It needs:

- `connectionString = ResourceId=<workspace GUID>`
- `container.name   = <lakehouse GUID>`
- `container.query  = <subfolder under Files/>`  *(no leading `Files/`)*

In [ ]:
data_source = {
    'name': DS_NAME,
    'type': 'onelake',
    'credentials': { 'connectionString': f'ResourceId={FABRIC_WORKSPACE_GUID}' },
    'container':   { 'name': LAKEHOUSE_GUID, 'query': LAKEHOUSE_SUBPATH },
}
put(f"/datasources('{DS_NAME}')", data_source)
print('Data source created:', DS_NAME)

## 3. Index — one document per chunk

Each chunk from Content Understanding becomes one search doc. `page_number` = `pageNumberFrom`, `page_to` = `pageNumberTo` (a chunk can span pages).

In [ ]:
index_def = {
    'name': IDX_NAME,
    'fields': [
        {'name': 'chunk_id',     'type': 'Edm.String', 'key': True, 'filterable': True, 'sortable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id',    'type': 'Edm.String', 'filterable': True},
        {'name': 'file_name',    'type': 'Edm.String', 'filterable': True, 'searchable': True},
        {'name': 'file_path',    'type': 'Edm.String', 'filterable': True},
        {'name': 'page_number',  'type': 'Edm.Int32',  'filterable': True, 'sortable': True, 'facetable': True},
        {'name': 'page_to',      'type': 'Edm.Int32',  'filterable': True, 'sortable': True},
        {'name': 'kind',         'type': 'Edm.String', 'filterable': True, 'facetable': True},
        {'name': 'image_path',   'type': 'Edm.String', 'filterable': True},
        {'name': 'content',      'type': 'Edm.String', 'searchable': True},
        {'name': 'content_vector','type': 'Collection(Edm.Single)', 'searchable': True,
         'dimensions': EMBED_DIMS, 'vectorSearchProfile': 'hnsw-aoai'},
    ],
    'vectorSearch': {
        'algorithms': [{'name': 'hnsw-algo', 'kind': 'hnsw'}],
        'profiles':   [{'name': 'hnsw-aoai', 'algorithm': 'hnsw-algo', 'vectorizer': 'aoai-vec'}],
        'vectorizers':[{'name': 'aoai-vec', 'kind': 'azureOpenAI',
            'azureOpenAIParameters': {
                'resourceUri':  AOAI_ENDPOINT,
                'deploymentId': AOAI_EMBED_DEPL,
                'apiKey':       AOAI_KEY,
                'modelName':    AOAI_EMBED_MODEL,
            }
        }]
    },
    'semantic': {
        'configurations': [{
            'name': 'sem-default',
            'prioritizedFields': {
                'titleField': {'fieldName': 'file_name'},
                'prioritizedContentFields': [{'fieldName': 'content'}]
            }
        }]
    }
}
put(f"/indexes('{IDX_NAME}')", index_def)
print('Index created:', IDX_NAME)

## 4. Skillset — Content Understanding + AOAI embedding

The Content Understanding skill **chunks the document itself** (no Text Split skill needed). Each chunk in `text_sections` carries `locationMetadata.pageNumberFrom` / `pageNumberTo` — but **only if you set `extractionOptions: ["locationMetadata"]`**. The index projection writes one search doc per chunk.

In [ ]:
skillset = {
    'name': SS_NAME,
    'description': 'CU extraction + GenAI image verbalization → AOAI embeddings',
    'cognitiveServices': {
        '@odata.type': '#Microsoft.Azure.Search.AIServicesByKey',
        'description': 'Azure AI Foundry resource (billing for CU + GenAI Prompt skills)',
        'key': FOUNDRY_KEY,
        'subdomainUrl': FOUNDRY_ENDPOINT,
    },
    'skills': [
        # 1) Content Understanding — extracts text chunks AND figure images.
        {
            '@odata.type': '#Microsoft.Skills.Util.ContentUnderstandingSkill',
            'name': 'cu',
            'description': 'Extract text, figures, and page metadata',
            'context': '/document',
            'extractionOptions': ['images', 'locationMetadata'],
            'chunkingProperties': {
                'unit': 'characters',
                'maximumLength': 2000,
                'overlapLength': 200
            },
            'inputs':  [{'name': 'file_data', 'source': '/document/file_data'}],
            'outputs': [
                {'name': 'text_sections',     'targetName': 'text_sections'},
                {'name': 'normalized_images', 'targetName': 'normalized_images'}
            ]
        },
        # 2) GenAI Prompt — verbalize each extracted figure (multimodal chat completion).
        {
            '@odata.type': '#Microsoft.Skills.Util.GenAIPromptSkill',
            'name': 'verbalize',
            'description': 'Generate a natural-language description of each figure',
            'context': '/document/normalized_images/*',
            'uri':           f'{AOAI_ENDPOINT}/openai/deployments/{AOAI_CHAT_DEPL}/chat/completions',
            'apiKey':        AOAI_KEY,
            'systemPrompt':  VERBALIZE_PROMPT,
            'userPrompt':    'Describe the figure.',
            'inputs':  [{'name': 'image', 'source': '/document/normalized_images/*/data'}],
            'outputs': [{'name': 'response', 'targetName': 'description'}]
        },
        # 3) Embed text chunks.
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'embed-text',
            'context': '/document/text_sections/*',
            'resourceUri':  AOAI_ENDPOINT,
            'apiKey':       AOAI_KEY,
            'deploymentId': AOAI_EMBED_DEPL,
            'modelName':    AOAI_EMBED_MODEL,
            'dimensions':   EMBED_DIMS,
            'inputs':  [{'name': 'text',      'source': '/document/text_sections/*/content'}],
            'outputs': [{'name': 'embedding', 'targetName': 'content_vector'}]
        },
        # 4) Embed figure descriptions (re-uses the same field name so retrieval can fuse them).
        {
            '@odata.type': '#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill',
            'name': 'embed-figure',
            'context': '/document/normalized_images/*',
            'resourceUri':  AOAI_ENDPOINT,
            'apiKey':       AOAI_KEY,
            'deploymentId': AOAI_EMBED_DEPL,
            'modelName':    AOAI_EMBED_MODEL,
            'dimensions':   EMBED_DIMS,
            'inputs':  [{'name': 'text',      'source': '/document/normalized_images/*/description'}],
            'outputs': [{'name': 'embedding', 'targetName': 'content_vector'}]
        }
    ],
    # Two projections into the SAME index. `kind` distinguishes them so retrieval can
    # filter or fuse text and figure hits.
    'indexProjections': {
        'selectors': [
            {
                'targetIndexName': IDX_NAME,
                'parentKeyFieldName': 'parent_id',
                'sourceContext': '/document/text_sections/*',
                'mappings': [
                    {'name': 'kind',          'source': '="text"'},
                    {'name': 'file_name',     'source': '/document/metadata_storage_name'},
                    {'name': 'file_path',     'source': '/document/metadata_storage_path'},
                    {'name': 'page_number',   'source': '/document/text_sections/*/locationMetadata/pageNumberFrom'},
                    {'name': 'page_to',       'source': '/document/text_sections/*/locationMetadata/pageNumberTo'},
                    {'name': 'content',       'source': '/document/text_sections/*/content'},
                    {'name': 'content_vector','source': '/document/text_sections/*/content_vector'}
                ]
            },
            {
                'targetIndexName': IDX_NAME,
                'parentKeyFieldName': 'parent_id',
                'sourceContext': '/document/normalized_images/*',
                'mappings': [
                    {'name': 'kind',          'source': '="figure"'},
                    {'name': 'file_name',     'source': '/document/metadata_storage_name'},
                    {'name': 'file_path',     'source': '/document/metadata_storage_path'},
                    {'name': 'page_number',   'source': '/document/normalized_images/*/locationMetadata/pageNumberFrom'},
                    {'name': 'page_to',       'source': '/document/normalized_images/*/locationMetadata/pageNumberTo'},
                    {'name': 'image_path',    'source': '/document/normalized_images/*/imagePath'},
                    {'name': 'content',       'source': '/document/normalized_images/*/description'},
                    {'name': 'content_vector','source': '/document/normalized_images/*/content_vector'}
                ]
            }
        ],
        'parameters': { 'projectionMode': 'skipIndexingParentDocuments' }
    }
}
put(f"/skillsets('{SS_NAME}')", skillset)
print('Skillset created:', SS_NAME)

## 5. Indexer

`allowSkillsetToReadFileData = true` is what exposes `/document/file_data` to the Content Understanding skill.

In [ ]:
indexer = {
    'name': IDXR_NAME,
    'dataSourceName':  DS_NAME,
    'targetIndexName': IDX_NAME,
    'skillsetName':    SS_NAME,
    'parameters': {
        'batchSize': 1,
        'maxFailedItems': 5,
        'configuration': {
            'dataToExtract': 'contentAndMetadata',
            'parsingMode': 'default',
            'allowSkillsetToReadFileData': True,
            'indexedFileNameExtensions': '.pdf,.docx,.pptx,.xlsx,.html,.txt,.md'
        }
    },
    'fieldMappings': [
        {'sourceFieldName': 'metadata_storage_path', 'targetFieldName': 'parent_id', 'mappingFunction': {'name': 'base64Encode'}}
    ]
}
put(f"/indexers('{IDXR_NAME}')", indexer)
post(f"/indexers('{IDXR_NAME}')/search.run")
print('Indexer started:', IDXR_NAME)

## 6. Watch status

In [ ]:
for _ in range(60):
    status = get(f"/indexers('{IDXR_NAME}')/search.status")
    last = status.get('lastResult') or {}
    print(status.get('status'), '|', last.get('status'),
          '| processed:', last.get('itemsProcessed'),
          '| failed:', last.get('itemsFailed'),
          '| errors:', (last.get('errors') or [])[:2])
    if last.get('status') in ('success', 'transientFailure', 'persistentFailure'):
        break
    time.sleep(15)

## 7. Query — verify page numbers come back

In [ ]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizableTextQuery

client = SearchClient(SEARCH_ENDPOINT, IDX_NAME, AzureKeyCredential(SEARCH_ADMIN_KEY))
results = client.search(
    search_text='termination clause',
    vector_queries=[VectorizableTextQuery(text='termination clause', k_nearest_neighbors=5, fields='content_vector')],
    select=['file_name', 'kind', 'page_number', 'content'],
    top=5,
)
for r in results:
    tag = '🖼️' if r['kind'] == 'figure' else '📄'
    print(f"{tag} {r['file_name']} p.{r['page_number']} ({r['kind']}): {r['content'][:160]}…")

## Cleanup (optional)

In [ ]:
for path in [f"/indexers('{IDXR_NAME}')", f"/skillsets('{SS_NAME}')", f"/indexes('{IDX_NAME}')", f"/datasources('{DS_NAME}')"]:
    r = requests.delete(f'{SEARCH_ENDPOINT}{path}?api-version={API_VERSION}', headers=HEADERS)
    print(path, '→', r.status_code)